In [80]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_percentage_error, r2_score

In [81]:
# --- Columns ---
y_col = "total_transmit_duration"
x_size = "total_transmit_size"
x_count = "transmit_count"

def visualize(train_df: pd.DataFrame, model: sm.OLS):
    # --- Build a prediction grid over size × count ---
    n_side = 50  # grid resolution per axis (increase for smoother surface)
    size_lin = np.linspace(train_df[x_size].min(), train_df[x_size].max(), n_side)
    count_lin = np.linspace(train_df[x_count].min(), train_df[x_count].max(), n_side)
    S, C = np.meshgrid(size_lin, count_lin)

    grid_df = pd.DataFrame({x_size: S.ravel(), x_count: C.ravel()})

    # Mean prediction
    mean_pred = model.get_prediction(grid_df).summary_frame(alpha=0.05)  # 95%
    Z_mean = mean_pred["mean"].values.reshape(S.shape)

    # 95% Prediction Interval (for new observations)
    Z_pi_low = mean_pred["obs_ci_lower"].values.reshape(S.shape)
    Z_pi_high = mean_pred["obs_ci_upper"].values.reshape(S.shape)

    # --- Build interactive 3D figure ---
    fig = go.Figure()

    # Fitted surface (mean)
    fig.add_trace(go.Surface(x=S, y=C, z=Z_mean, name="Fitted surface (mean)", showscale=False, opacity=0.85))

    # Raw data points
    fig.add_trace(
        go.Scatter3d(
            x=train_df[x_size],
            y=train_df[x_count],
            z=train_df[y_col],
            mode="markers",
            name="Data",
            marker=dict(size=3, opacity=0.6),
        )
    )

    # Optional: lower PI surface (toggle via legend)
    fig.add_trace(
        go.Surface(x=S, y=C, z=Z_pi_low, name="Lower 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    )

    # Optional: upper PI surface (toggle via legend)
    fig.add_trace(
        go.Surface(x=S, y=C, z=Z_pi_high, name="Upper 95% PI", showscale=False, opacity=0.25, visible="legendonly")
    )

    fig.update_layout(
        title="Interactive 3D: Duration ~ Size + Transmit Count",
        scene=dict(
            xaxis_title="Total Transmit Size",
            yaxis_title="Transmit Count",
            zaxis_title="Total Transmit Duration",
            camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)),
        ),
        legend=dict(itemsizing="constant"),
    )

    fig.show()

In [82]:
def evaluate(y_true, y_pred):
    print("R² =", r2_score(y_true, y_pred))
    print("MAPE =", mean_absolute_percentage_error(y_true, y_pred))

    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_pred) + np.abs(y_true)))
    print(f"SMAPE = {smape:.2f}%")


def fit_model(train_df: pd.DataFrame):
    # --- Fit OLS: duration ~ size + count ---
    X = train_df[[x_size, x_count]].copy()
    y = train_df[y_col].copy()
    model = sm.OLS(y, X).fit()

    print(model.params)

    # Optional: quick coefficients readout
    print(model.summary().tables[1])

    # Predictions
    y_pred = model.predict(X) 
    evaluate(y, y_pred)

    # # Compute relative % error
    # rel_error = (y_pred - y_true) / y_true * 100
    # symmetric_err = 100 * (y_pred - y_true).abs() / ((y_pred.abs() + y_true.abs()) / 2)
    # # Put into a table
    # error_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "rel_error_%": rel_error, "symmetric_err_%": symmetric_err})
    # display(error_df.sort_values(by="y_true"))

    return model

In [ ]:
import json

dfs = []

for i in [2, 4, 8]:
    with open(f'../results/quest/tp_nccl_{i}.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"] * (i - 1)
    df.loc[:, "transmit_count"] = df["transmit_count"] * (i - 1)
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"] * i
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

print(df.size)

model = fit_model(df)
evaluate(df[y_col], model.predict(df[[x_size, x_count]]))

visualize(df, model)

    # X = df[[x_size, x_count]].copy()
    # y = df[y_col].copy()

    # # Model = a * x_count = b * x_size
    # # a = latency per transmit
    # # b = bandwidth inverse

    # a = 0.0001
    # b = 1 / 400e12

    # y_pred = a * X[x_count] + b * X[x_size]
    # y_true = y

    # evaluate(y_true, y_pred)

1134
total_transmit_size    7.371352e-11
transmit_count         3.344176e-05
dtype: float64
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
total_transmit_size  7.371e-11   1.18e-11      6.253      0.000    5.05e-11    9.69e-11
transmit_count       3.344e-05   9.66e-07     34.631      0.000    3.15e-05    3.53e-05
R² = 0.6487714262995696
MAPE = 0.4206761071277648
SMAPE = 57.99%
R² = 0.6487714262995696
MAPE = 0.4206761071277648
SMAPE = 57.99%


In [84]:
dfs = []

for i in [2, 4, 8]:
    with open(f'../results/quest/tp_gloo_{i}.json', 'r') as f:
        tp_data = json.load(f)

    df = pd.DataFrame(tp_data)
    df = df[["total_transmit_duration", "total_transmit_size", "transmit_count"]]
    df.loc[:, "total_transmit_size"] = df["total_transmit_size"] * (i - 1)
    df.loc[:, "transmit_count"] = df["transmit_count"] * (i - 1)
    df.loc[:, "total_transmit_duration"] = df["total_transmit_duration"]
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

print(df.size)

model = fit_model(df)
evaluate(df[y_col], model.predict(df[[x_size, x_count]]))

visualize(df, model)

    # X = df[[x_size, x_count]].copy()
    # y = df[y_col].copy()

    # # Model = a * x_count = b * x_size
    # # a = latency per transmit
    # # b = bandwidth inverse

    # a = 0.0001
    # b = 1 / 400e12

    # y_pred = a * X[x_count] + b * X[x_size]
    # y_true = y

    # evaluate(y_true, y_pred)

1134
total_transmit_size    4.656396e-09
transmit_count         6.358998e-04
dtype: float64
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
total_transmit_size  4.656e-09   1.24e-10     37.523      0.000    4.41e-09     4.9e-09
transmit_count          0.0006   1.02e-05     62.598      0.000       0.001       0.001
R² = 0.9266065102384846
MAPE = 0.24536384535244052
SMAPE = 30.00%
R² = 0.9266065102384846
MAPE = 0.24536384535244052
SMAPE = 30.00%
